<a href="https://colab.research.google.com/github/ravi-1718/Delta/blob/main/workshop/Helmet_Detection_Participant_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deploying Vision AI at the Edge
## Hands-On Helmet Detection

**Time needed:** 2.5 to 3 hours
**You need:** a laptop, a browser and a Google account
**Special hardware:** not needed

By the end, you will have built this system, step by step:

```text
Traffic video
    |
YOLO model finds objects in each frame
    |
ByteTrack gives each object an ID that stays the same
    |
Match each rider to a motorcycle, and each head to a rider
    |
Use many frames to decide: helmet or no helmet?
    |
Save a video with the result drawn on it
    |
Convert the model to ONNX, run it on CPU, measure the speed
```

### Labels used in this notebook

- 🟩 **RUN** — just run the cell. Do not change it.
- 👀 **OBSERVE** — read the output and think about the question.
- ✏️ **TRY** — change one value and see what happens.
- ⏭️ **OPTIONAL** — skip this if we are short of time.

> First, click **Runtime → Change runtime type → T4 GPU**. The notebook also works
> on CPU, but it will be about ten times slower.
>
> If you want to keep your changes, click **File → Save a copy in Drive**.
> Run the cells in order, from top to bottom.


## What we will build, and when

| Stage | What you will have at the end | Time | Skip if late |
|---|---|---:|---|
| 1. Setup | Colab ready to run | 15 min | — |
| 2. Detection | One frame with boxes drawn on it | 15 min | Exercise 1 |
| 3. Tracking | IDs that stay with each rider | 15 min | — |
| 4. Matching | Rider joined to motorcycle and helmet | 25 min | the TRY step |
| 5. Decision | Four possible answers per rider | 20 min | — |
| 6. Full system | A video with the results drawn on it | 20 min | the download cell |
| 7. Edge model | ONNX model, speed test and report | 30 min | quantization part |
| 8. Review | Discussion | 10 min | — |

Parts marked **Core** are done by everyone together. Parts marked **Explore**
are for you to try later at home. Skipping them now costs you nothing.


# 1. Setup — Core

*Time: 15 min.*

We need two files: the trained model, and a short video clip. The rest of the
system we will write ourselves, step by step.

Before you start, click **Runtime → Change runtime type → T4 GPU**.
A GPU is not compulsory, but the notebook runs much faster with one.

### Why we print versions and file paths

The same model can give different results with a different version of a
library. So we print the version numbers. If something goes wrong later, you
can check whether a version changed.

We also keep all file paths in one place. Later, when you move this code to a
Jetson board or your own laptop, you only have to change those few lines.


In [ ]:
# 🟩 RUN — Install the workshop packages
# Install tested workshop dependencies.
# If Colab asks for a runtime restart after installation, restart and rerun from here.
# lap (tracking) and onnxslim (export) are pulled in by Ultralytics at the
# moment they are first needed. Installing them here keeps that from
# happening mid-session, which otherwise prints a restart warning while
# everyone is waiting.
!pip -q install "ultralytics>=8.3,<9" "onnx>=1.17,<2" "onnxruntime>=1.20,<2" "scipy>=1.13,<2" "lap>=0.5.12" "onnxslim>=0.1.82"

In [ ]:
# 🟩 RUN — Download the workshop assets
from pathlib import Path
import os
import subprocess
import time
import urllib.request
import urllib.error

# The workshop needs exactly two files: the trained weights and one short
# clip. Fetching them directly transfers about 53 MB. Cloning the whole
# project repository would transfer several hundred MB of duplicated
# models and videos that this notebook never opens.
RAW_BASE = "https://raw.githubusercontent.com/EmertxeInfoTech/edge-ai-helmet-detection-workshop/main/final"

# Colab provides /content; locally the assets land beside this notebook.
PROJECT_ROOT = Path("/content/workshop") if Path("/content").exists() else Path("workshop")
MODEL_PATH = PROJECT_ROOT / "models" / "best.pt"
VIDEO_PATH = PROJECT_ROOT / "sample_videos" / "sample.mp4"
OUTPUT_DIR = PROJECT_ROOT / "workshop_output"

for directory in (MODEL_PATH.parent, VIDEO_PATH.parent, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def fetch(url, destination, expected_mb, attempts=2):
    """Download once. Rerunning this cell reuses a complete local file."""
    # A partial file from an interrupted download is smaller than expected,
    # so size is checked rather than mere existence.
    if destination.exists() and destination.stat().st_size > 0.9 * expected_mb * 1e6:
        print(f"present     {destination.name:<12} {destination.stat().st_size / 1e6:>6.1f} MB")
        return
    for attempt in range(1, attempts + 1):
        try:
            started = time.time()
            urllib.request.urlretrieve(url, destination)
            print(f"downloaded  {destination.name:<12} "
                  f"{destination.stat().st_size / 1e6:>6.1f} MB in {time.time() - started:.0f}s")
            return
        except urllib.error.URLError as error:
            print(f"attempt {attempt} failed: {error}")
    raise RuntimeError(f"Could not download {destination.name}. Ask the instructor for the USB copy.")


fetch(f"{RAW_BASE}/models/best.pt", MODEL_PATH, 23)
fetch(f"{RAW_BASE}/sample_videos/sample.mp4", VIDEO_PATH, 31)

print("\nModel: ", MODEL_PATH)
print("Video: ", VIDEO_PATH)
print("Output:", OUTPUT_DIR)


In [ ]:
# 🟩 RUN — Load the libraries and find out what hardware we have
# platform tells us the Python version, which we print for the record.
import platform
# OpenCV reads video files and draws boxes and text on images.
import cv2
# NumPy gives us fast arrays and the maths helpers we use for matching.
import numpy as np
# Matplotlib displays images inside the notebook.
import matplotlib.pyplot as plt
# PyTorch is the library the model itself runs on.
import torch
# Ultralytics provides the YOLO class: detection and tracking in one package.
import ultralytics

# torch.cuda.is_available() is True only when a GPU is attached to this
# session. Ultralytics expects the GPU number (0) or the string "cpu", so we
# store whichever applies and pass DEVICE to every call from here on.
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# Print the versions. If your results ever differ from a classmate's, these
# numbers are the first thing to compare.
print("Python:", platform.python_version())
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Execution device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


### 🟩 RUN — Check that everything is ready

This cell checks that all the files and tools are present, *before* we spend
time running the model. Every line must say `OK`. If any line says `FAILED`,
stop and ask for help.


In [ ]:
# 🟩 RUN — Check everything is in place before we run the model
# shutil.which() tells us whether a command line tool exists on this machine.
import shutil
# ONNX Runtime is the engine that will run our exported model in section 7.
import onnxruntime as ort

# Each entry is a question with a True or False answer.
preflight = {
    # Did the model file download completely?
    "trained model": MODEL_PATH.exists(),
    # Did the video download completely?
    "sample video": VIDEO_PATH.exists(),
    # Does the output folder exist, and are we allowed to write into it?
    "output directory": OUTPUT_DIR.exists() and os.access(OUTPUT_DIR, os.W_OK),
    # FFmpeg converts our video so a browser can play it.
    "FFmpeg": shutil.which("ffmpeg") is not None,
    # Is ONNX Runtime installed and importable?
    "ONNX Runtime": bool(ort.__version__),
}

# Print one line per check so a failure is easy to spot.
for item, ready in preflight.items():
    print(f"{'OK' if ready else 'FAILED':<7} {item}")

# assert stops the notebook here if any check failed. Failing now, with a clear
# message, is much better than failing later inside the model.
assert all(preflight.values()), "Preflight failed. Read the troubleshooting section before continuing."
print("\n✅ Workshop environment is ready")


### Checkpoint 1

You should see this:

```text
OK      trained model
OK      sample video
OK      output directory
OK      FFmpeg
OK      ONNX Runtime
✅ Workshop environment is ready
```

Your Python version, library versions and CPU or GPU name may be different
from the instructor's. That is fine.


# 2. First look: what does the model see? — Core

*Time: 15 min.*

**Inference** means running the trained model on an image and getting its
answer back. Our model was trained to find five kinds of object:

| ID | Class | What it means |
|---:|---|---|
| 0 | rider | A person sitting on the motorcycle |
| 1 | motorcycle | The motorcycle itself |
| 2 | helmet | A head with a helmet on it |
| 3 | no_helmet | A head with no helmet |
| 4 | license_plate | The number plate |

The model tells us **what** each object is and **where** it is. It does not
tell us which helmet belongs to which rider, or which rider is sitting on
which motorcycle. We will have to work that out ourselves. That is the main
job of this workshop.


In [ ]:
# 🟩 RUN — Load the model and run it on a single frame
# The YOLO class loads the trained weights and runs them for us.
from ultralytics import YOLO

# Loading reads the file from disk into memory. We do it once and reuse it.
model = YOLO(str(MODEL_PATH))

# The model reports a number for each object it finds. This table turns those
# numbers into names we can read. The order matters and must match the model.
CLASS_NAMES = {0: "rider", 1: "motorcycle", 2: "helmet", 3: "no_helmet", 4: "license_plate"}

# Open the video file so we can pull out one frame to experiment with.
cap = cv2.VideoCapture(str(VIDEO_PATH))
# Jump to frame 20. The first frames of a clip are often less interesting.
cap.set(cv2.CAP_PROP_POS_FRAMES, 20)
# read() returns two things: whether it worked, and the image itself.
ok, sample_frame = cap.read()
# Close the file as soon as we have what we need.
cap.release()
# Stop with a clear message if the frame could not be read.
assert ok, "Could not read the sample video"

# Now run the model on that one frame.
#   conf=0.25   report an object only if the model is at least 25% sure
#   imgsz=640   resize the frame to 640 pixels before running the model
#   device      the GPU or CPU we chose earlier
#   verbose     False keeps the output clean; set it to True to see more
# predict() returns a list with one entry per image, so we take entry [0].
result = model.predict(sample_frame, conf=0.25, imgsz=640, device=DEVICE, verbose=False)[0]

print(f"Detected {len(result.boxes)} objects")
# The shape is (height, width, colour channels). This clip is 1440x2560.
print("Frame shape:", sample_frame.shape)


In [ ]:
# 🟩 RUN — Draw the result on screen
def show_bgr(image, title="", figsize=(14, 8)):
    """Display an OpenCV image inside the notebook.

    OpenCV stores colours in the order blue, green, red. Matplotlib expects
    red, green, blue. Without the conversion below, everything looks blue.
    """
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    # Pixel numbers along the edges add nothing here, so hide them.
    plt.axis("off")
    plt.show()

# result.plot() returns a copy of the frame with the boxes already drawn.
show_bgr(result.plot(), "The model's output on one traffic frame")


In [ ]:
# 🟩 RUN — Look at the actual numbers behind the picture
# A picture is convincing, but the model really returns numbers. Print them.
print(f"{'class':<16} {'confidence':>10}  box [x1, y1, x2, y2]")
print("-" * 65)

# Each detection has three parts, and zip() walks through them together:
#   cls   the class number  (0 to 4)
#   conf  how sure the model is  (0 to 1)
#   box   where it is, as four pixel coordinates
for cls, conf, box in zip(result.boxes.cls, result.boxes.conf, result.boxes.xyxy):
    # Translate the class number into a readable name.
    name = CLASS_NAMES[int(cls)]
    # x1,y1 is the top-left corner and x2,y2 the bottom-right corner.
    # Round them so the table stays easy to read.
    coords = [round(v, 1) for v in box.tolist()]
    print(f"{name:<16} {float(conf):>10.2f}  {coords}")


### Exercise 1 — Explore

In the cell above, change `conf=0.25` to `conf=0.50` and run it again.
`conf` is the confidence cut-off: the model only reports an object when it is
at least this sure.

Then answer:

- How many boxes disappeared?
- Were all the boxes that disappeared actually wrong?
- So is a higher cut-off always better?

### Checkpoint 2

Make sure you can explain these three words to the person next to you:

- **class** — what the object is (rider, helmet, ...)
- **bounding box** — where it is, as four numbers: `[x1, y1, x2, y2]`
- **confidence** — how sure the model is, from 0 to 1


### 👀 OBSERVE — What you should see

- Boxes around riders, motorcycles, heads and number plates.
- A printed table with one row per box.
- Your exact numbers will be a little different from your neighbour's. That is
  normal, and happens because of library and hardware differences.

If the model finds **zero** objects, something is wrong. Stop and ask for help.


# 3. Tracking: keeping the same ID across frames — Core

*Time: 15 min.*

The model looks at each frame on its own. It has no memory. In frame 20 it
sees "a rider", and in frame 21 it again sees "a rider" — but it does not know
that this is the *same* person.

A **tracker** solves this. It gives each object a number, called a track ID,
and tries to keep that number with the same object as it moves.

We need this. One frame is not enough to judge anyone: the photo may be
blurred, or a bus may pass in front of the rider. We want to collect many
views of the same rider and then decide. Without IDs, we cannot tell which
views belong to the same person.


In [ ]:
# 🟩 RUN — Track the objects and compare two frames in a row
# Load a second copy of the model for tracking, so the earlier single-frame
# example stays untouched and you can go back to it.
tracking_model = YOLO(str(MODEL_PATH))

# track() runs detection on every frame AND links objects between frames.
track_stream = tracking_model.track(
    source=str(VIDEO_PATH),
    # ByteTrack is the tracking method. It matches boxes between frames.
    tracker="bytetrack.yaml",
    # persist=True keeps the tracker's memory between frames, which is what
    # allows an ID to stay with the same rider.
    persist=True,
    # stream=True hands us one frame at a time instead of building a huge
    # list of every frame in memory first.
    stream=True,
    # Only look for our five classes.
    classes=list(CLASS_NAMES),
    conf=0.25,
    imgsz=640,
    device=DEVICE,
    verbose=False,
)

# We only want two frames for this demonstration, 20 and 21.
tracked_results = []
for frame_no, tracked in enumerate(track_stream):
    if frame_no in (20, 21):
        tracked_results.append(tracked)
    # Stop as soon as we have both. There is no need to process the rest.
    if frame_no >= 21:
        break

# Show the two frames side by side and compare the ID numbers on the boxes.
fig, axes = plt.subplots(1, 2, figsize=(20, 7))
for ax, tracked, frame_no in zip(axes, tracked_results, (20, 21)):
    ax.imshow(cv2.cvtColor(tracked.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {frame_no}: the same rider keeps the same ID")
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# 🟩 RUN — Reshape the tracker output into something we can work with
def unpack_tracked_result(r):
    """Turn one frame of tracker output into two lookup tables.

    Ultralytics gives us four parallel lists: classes, IDs, boxes and
    confidences. Searching through parallel lists every time we need
    something is painful, so we rearrange them once, here, into:

        detections  = {"rider": {track_id: box}, "helmet": {...}, ...}
        confidences = {"rider": {track_id: confidence}, ...}

    Every later step in this notebook reads these two dictionaries.
    """
    # Start with one empty dictionary per class, so a class that appears in
    # no frame still exists as a key and nothing has to check for it later.
    detections = {name: {} for name in CLASS_NAMES.values()}
    confidences = {name: {} for name in CLASS_NAMES.values()}

    # When the tracker finds nothing at all, r.boxes.id is None rather than an
    # empty list. Return the empty tables in that case.
    if r.boxes.id is None:
        return detections, confidences

    # Walk the four lists together, one detection per loop.
    for cls, tid, box, conf in zip(
        r.boxes.cls.tolist(), r.boxes.id.tolist(),
        r.boxes.xyxy.tolist(), r.boxes.conf.tolist()
    ):
        # Class number to name, e.g. 0 becomes "rider".
        name = CLASS_NAMES[int(cls)]
        # File the box and the confidence under that class and track ID.
        detections[name][int(tid)] = box
        confidences[name][int(tid)] = float(conf)
    return detections, confidences

# Try it on the first of the two frames we kept.
tracked_detections, tracked_confidences = unpack_tracked_result(tracked_results[0])
for name, objects in tracked_detections.items():
    print(f"{name:<15}: {len(objects):>2} objects — IDs {list(objects)}")


### Checkpoint 3

- Find one rider ID that appears in both of the frames shown above.
- What could make the tracker give the same person a new ID?
- A track ID is not a person's name or identity. Why is that an important
  difference for a system like this?


# 4. Matching: which objects belong together? — Core

*Time: 25 min.*

After tracking we have a list of boxes with IDs. But a list is not enough.
To say "this rider has no helmet", we must join objects together:

1. rider ↔ motorcycle
2. head (helmet or no_helmet) ↔ rider
3. number plate ↔ motorcycle

The model cannot do this for us. We do it using **geometry** — that is, using
where the boxes are on the screen.

Think about how you would do it by eye. A rider sits on top of a motorcycle,
so their boxes overlap a lot. A head is at the top of the rider's box. A
number plate is at the bottom of the motorcycle.

We turn that idea into numbers, and then let an algorithm pick the best set of
pairs. The rule we use is called the **Hungarian algorithm**. It looks at all
possible pairs at once and picks the combination with the lowest total cost.
After it chooses, we still throw away any pair that looks too unlikely.


In [ ]:
# 🟩 RUN — Measurements we need for matching
# linear_sum_assignment is SciPy's implementation of the Hungarian algorithm.
# Give it a table of costs and it picks the best set of pairs.
from scipy.optimize import linear_sum_assignment


def center(box):
    """The middle point of a box given as [x1, y1, x2, y2]."""
    x1, y1, x2, y2 = box
    # Average of the two x values, and of the two y values.
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def top_center(box):
    """The middle of the top edge of a box.

    For a rider box this is roughly where the head sits, so it is the point
    we compare against helmet boxes. Using the middle of the whole rider box
    would put us near the chest and match the wrong head in a crowd.
    """
    x1, y1, x2, _ = box
    return ((x1 + x2) / 2, y1)


def bottom_center(box):
    """The middle of the bottom edge of a box.

    For a motorcycle this is where the number plate usually is.
    """
    x1, _, x2, y2 = box
    return ((x1 + x2) / 2, y2)


def iou(box_a, box_b):
    """How much two boxes overlap, as a number from 0 to 1.

    IoU means Intersection over Union:

        the area the two boxes share  /  the total area they cover together

    0.0 means they do not touch at all. 1.0 means they are exactly the same
    box. A rider sitting on a motorcycle usually gives something like 0.3-0.6.
    """
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    # The shared rectangle starts at the rightmost of the two left edges and
    # the lowest of the two top edges...
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    # ...and ends at the leftmost of the two right edges and the highest of
    # the two bottom edges.
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)

    # If the boxes do not overlap, these differences go negative, and max(0, ...)
    # turns the area into 0 instead of a meaningless positive number.
    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)

    # Area of each box on its own.
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    # The shared part is inside both areas, so subtract it once or it counts twice.
    union = area_a + area_b - intersection

    # Guard against dividing by zero when both boxes have no area.
    return intersection / union if union else 0.0


def point_distance(point_a, point_b):
    """Straight-line distance between two points, in pixels.

    np.hypot(dx, dy) computes sqrt(dx*dx + dy*dy) for us.
    """
    return float(np.hypot(point_a[0] - point_b[0], point_a[1] - point_b[1]))


In [ ]:
# 🟩 RUN — Match riders to motorcycles, and heads to riders
def associate_riders_to_motorcycles(riders, motorcycles, max_cost=0.90):
    """Decide which rider is sitting on which motorcycle.

    Returns {rider_id: motorcycle_id}.

    The idea: a rider and their motorcycle overlap a lot, so a high IoU means
    a likely pair. The Hungarian algorithm needs a *cost* to make small, not a
    score to make big, so we use cost = 1 - IoU. High overlap becomes low cost.
    """
    # Nothing to match if either list is empty.
    if not riders or not motorcycles:
        return {}

    # Fix an order for the IDs, because the cost table is addressed by position.
    rider_ids, motorcycle_ids = list(riders), list(motorcycles)

    # Build the cost table: one row per rider, one column per motorcycle.
    # Entry [r][c] is the cost of pairing rider r with motorcycle c.
    costs = np.array([
        [1.0 - iou(riders[rid], motorcycles[mid]) for mid in motorcycle_ids]
        for rid in rider_ids
    ])

    # The Hungarian algorithm picks one motorcycle per rider so that the total
    # cost is as low as possible. It looks at the whole table at once, so it
    # avoids the trap of grabbing the best-looking pair first and leaving a bad
    # pairing for later riders.
    rows, cols = linear_sum_assignment(costs)

    # The algorithm always returns its best attempt, even when that attempt is
    # poor. max_cost is our veto: pairs worse than this are thrown away.
    return {
        rider_ids[r]: motorcycle_ids[c]
        for r, c in zip(rows, cols)
        if costs[r, c] <= max_cost
    }


def associate_heads_to_riders(riders, helmets, no_helmets,
                               helmet_confs, no_helmet_confs,
                               max_distance=60):
    """Decide which head belongs to which rider, and whether it wore a helmet.

    Returns {rider_id: (status, confidence)} where status is "helmet" or
    "no_helmet".
    """
    # Put helmet and no-helmet boxes into ONE pool of candidate heads. A rider
    # has one head, so these two classes must compete against each other. If we
    # matched them separately, the same rider could be given a helmet and a
    # no-helmet at the same time.
    heads = {
        **{hid: (box, "helmet", helmet_confs[hid]) for hid, box in helmets.items()},
        **{hid: (box, "no_helmet", no_helmet_confs[hid]) for hid, box in no_helmets.items()},
    }
    if not riders or not heads:
        return {}

    rider_ids, head_ids = list(riders), list(heads)

    # Here the cost is distance in pixels: from the top of the rider box (where
    # a head should be) to the centre of the head box. Closer means cheaper.
    costs = np.array([
        [point_distance(top_center(riders[rid]), center(heads[hid][0])) for hid in head_ids]
        for rid in rider_ids
    ])

    rows, cols = linear_sum_assignment(costs)

    matches = {}
    for r, c in zip(rows, cols):
        # Reject a match if the head is more than max_distance pixels away.
        # Without this, a lone rider would be matched to a helmet on the far
        # side of the road simply because nothing closer was available.
        if costs[r, c] <= max_distance:
            _, status, confidence = heads[head_ids[c]]
            matches[rider_ids[r]] = (status, confidence)
    return matches


In [ ]:
# 🟩 RUN — Try the matching on one real frame
# Reshape frame 20 into the two lookup tables.
det, confs = unpack_tracked_result(tracked_results[0])

# Who is on which motorcycle?
rider_motorcycle = associate_riders_to_motorcycles(det["rider"], det["motorcycle"])

# Which head belongs to each rider, and was it wearing a helmet?
head_rider = associate_heads_to_riders(
    det["rider"], det["helmet"], det["no_helmet"],
    confs["helmet"], confs["no_helmet"]
)

# Print the pairs, with the overlap that justified each one.
print("Rider → motorcycle")
for rider_id, motorcycle_id in rider_motorcycle.items():
    overlap = iou(det["rider"][rider_id], det["motorcycle"][motorcycle_id])
    print(f"  rider {rider_id:>4} → motorcycle {motorcycle_id:>4}  IoU={overlap:.2f}")

# Note how few riders appear here compared to the list above. Most heads are
# too far away, too small or hidden, so no head could be matched to them.
print("\nRider → head status")
for rider_id, (status, confidence) in head_rider.items():
    print(f"  rider {rider_id:>4} → {status:<10} confidence={confidence:.2f}")


In [ ]:
# 🟩 RUN — Draw the matches so we can check them by eye
def draw_associations(image, detections, rider_motorcycle, head_rider):
    """Draw each rider, colour it by helmet status, and join it to its motorcycle."""
    # Work on a copy. OpenCV draws straight into the array it is given, and we
    # do not want to damage the original frame.
    canvas = image.copy()

    for rider_id, rider_box in detections["rider"].items():
        # Drawing needs whole numbers, not decimals.
        x1, y1, x2, y2 = map(int, rider_box)

        # Look up this rider's head status. "unknown" when no head matched.
        status = head_rider.get(rider_id, ("unknown", 0))[0]

        # Colours are in blue-green-red order, because that is what OpenCV uses.
        # red = no helmet, green = helmet, orange = we could not tell.
        color = (0, 0, 255) if status == "no_helmet" else (0, 200, 0) if status == "helmet" else (0, 165, 255)

        # The box around the rider.
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)

        # The label above the box. max(18, ...) keeps the text on screen when
        # the rider is near the top edge of the frame.
        cv2.putText(canvas, f"R{rider_id}: {status}", (x1, max(18, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        # A yellow line from rider to motorcycle shows the pairing we computed.
        if rider_id in rider_motorcycle:
            motorcycle_id = rider_motorcycle[rider_id]
            rc = tuple(map(int, center(rider_box)))
            mc = tuple(map(int, center(detections["motorcycle"][motorcycle_id])))
            cv2.line(canvas, rc, mc, (255, 255, 0), 2)
    return canvas


# Check the lines by eye: does each one join a rider to the machine they are
# actually sitting on?
associated_frame = draw_associations(
    tracked_results[0].orig_img, det, rider_motorcycle, head_rider
)
show_bgr(associated_frame, "Helmet status, and which rider is on which motorcycle")


### ✏️ TRY — Change one number and watch it break

Find the call to `associate_riders_to_motorcycles` and change only `max_cost`:

```python
max_cost=1.0   # accept every pair, even boxes that do not overlap at all
max_cost=0.5   # accept a pair only if the boxes overlap by at least 50%
```

Run the cell again after each change and look at the picture.

- With `1.0`, do you see lines joining riders to the wrong motorcycle?
- With `0.5`, do some correct pairs disappear?

This is the trade-off you will meet again and again. A loose rule accepts
wrong answers. A strict rule throws away right answers. Put the value back to
`0.90` before you continue.


# 5. Deciding, using many frames — Core

*Time: 20 min.*

One frame is not proof. The photo may be blurred, the head may be half hidden,
or the model may simply be wrong for a moment.

So for every rider we keep a list of what we saw in each frame. Only then do
we decide. We accept an answer only when all three of these are true:

1. we have seen the rider in **enough frames**,
2. **most** of those frames agree with each other, and
3. the model was **confident enough** on average.

This gives four possible answers, not two:

| Answer | What it means | Colour in the video |
|---|---|---|
| `pending` | We have not seen this rider enough times yet | Orange |
| `uncertain` | We have seen enough, but the evidence is weak or mixed | Yellow |
| `compliant` | Enough good evidence of a helmet | Green |
| `violation` | Enough good evidence of no helmet | Red |

Notice that two of the four answers are "I do not know yet" and "I am not
sure". A system that must always answer either *helmet* or *no helmet* will
be confidently wrong quite often.

Riders with no head evidence at all get no label drawn. It is better to stay
silent than to fill the video with wrong guesses.


In [ ]:
# 🟩 RUN — The rule that turns many frames into one answer
# Counter counts how often each value appears in a list.
# defaultdict lets us append to a list without creating it first.
from collections import Counter, defaultdict

# The three numbers that decide everything. Every one of them is a choice a
# human made, not something the model learned. You will change them later.
MIN_OBSERVATIONS = 3      # we must have seen the rider in at least 3 frames
MIN_AGREEMENT = 0.67      # at least 67% of those frames must say the same thing
MIN_CONFIDENCE = 0.70     # the model's average confidence must be at least 0.70


def decide_status(observations):
    """Turn a list of per-frame observations into one answer, with a reason.

    observations is a list of tuples, one per frame, like:
        [("no_helmet", 0.81), ("no_helmet", 0.77), ("helmet", 0.55)]

    We return a dictionary, not just a word, so that every answer can explain
    itself. Being able to say *why* matters: if this ever became a fine sent
    to a real person, "the computer said so" is not an acceptable reason.
    """
    # Rule 1: have we seen enough frames?
    # If not, we do not guess. We say "pending", meaning ask me later.
    if len(observations) < MIN_OBSERVATIONS:
        return {
            "verdict": "pending",
            "observations": len(observations),
            "reason": "not enough frames yet",
        }

    # Take just the labels, dropping the confidences: ["no_helmet", "helmet", ...]
    labels = [label for label, _ in observations]

    # Which label appears most often, and how many times?
    # most_common(1) returns [("no_helmet", 2)], so [0] unpacks that pair.
    majority_label, majority_count = Counter(labels).most_common(1)[0]

    # Collect the confidences that belong to the winning label only. Averaging
    # all confidences would mix in the frames that disagreed.
    majority_confidences = [conf for label, conf in observations if label == majority_label]

    # What fraction of frames agreed with the winner? 1.0 means all of them.
    agreement = majority_count / len(observations)

    # How sure was the model, on average, in those agreeing frames?
    detector_confidence = float(np.mean(majority_confidences))

    # Rule 2: did enough of the frames agree with each other?
    if agreement < MIN_AGREEMENT:
        verdict, reason = "uncertain", "the frames do not agree enough"
    # Rule 3: was the model confident enough in those frames?
    elif detector_confidence < MIN_CONFIDENCE:
        verdict, reason = "uncertain", "the model was not confident enough"
    # Both rules passed, so we accept the majority label as the answer.
    else:
        verdict = "violation" if majority_label == "no_helmet" else "compliant"
        reason = "accepted"

    # Return the answer together with every number behind it.
    return {
        "verdict": verdict,
        "majority": majority_label,
        "agreement": agreement,
        "detector_confidence": detector_confidence,
        "observations": len(observations),
        "reason": reason,
    }


In [ ]:
# 🟩 RUN — Test the rule on cases we invented ourselves
# Before running this on real video, feed it examples where we already know
# what the answer should be. This is how you check logic you have just written.
test_cases = {
    # Only one frame. Strong, but not enough of them yet.
    "one strong frame": [("no_helmet", 0.95)],
    # Three frames, all agreeing, all confident. This should be accepted.
    "consistent violation": [("no_helmet", 0.91), ("no_helmet", 0.88), ("no_helmet", 0.94)],
    # Three frames agreeing, but the model was unsure every time.
    "low-confidence violation": [("no_helmet", 0.55), ("no_helmet", 0.61), ("no_helmet", 0.58)],
    # Enough frames and high confidence, but they contradict each other.
    "mixed evidence": [("no_helmet", 0.93), ("helmet", 0.85), ("no_helmet", 0.91)],
    # Three clear helmet frames.
    "compliant": [("helmet", 0.89), ("helmet", 0.92), ("helmet", 0.87)],
}

# Read each line and check it against what you expected before running.
for name, observations in test_cases.items():
    print(f"{name:<26} → {decide_status(observations)}")


### 👀 OBSERVE — Why four answers and not two?

| Answer | Meaning |
|---|---|
| `PENDING` | Not enough frames yet. Ask me later. |
| `UNCERTAIN` | I have frames, but they do not agree, or the model was unsure. |
| `COMPLIANT` | Good evidence of a helmet. |
| `VIOLATION` | Good evidence of no helmet. |

The system is allowed to say "not yet" and "I am not sure". This matters a
lot here. If this output is later used to fine somebody, a wrong answer has a
real cost for a real person.


# 6. Putting it all together — Core

*Time: 20 min.*

Now we join every step: find objects, track them, match them, and decide.
Each frame is drawn with the current answer for every rider, and saved into a
video.

The cell below processes the whole clip: 500 frames, which is 20 seconds of
video. On a GPU this takes about a minute. On CPU it takes a few minutes, so
the cell prints its progress as it goes.

Why process all 500 frames? Because most riders who end as `pending` or
`uncertain` appear in the later part of the clip. A video that showed only
confirmed violations would give you a wrong idea of how the system behaves.

If we are running late, set `FAST_PREVIEW = True` in the cell. It will then
use only the first 150 frames.


In [ ]:
# 🟩 RUN — Draw one frame of the finished system
def draw_pipeline_frame(image, detections, rider_motorcycle, decisions):
    """Draw every rider with the colour of its current answer."""
    # Draw on a copy so the original frame stays clean.
    canvas = image.copy()

    # One colour per answer, in OpenCV's blue-green-red order.
    colors = {
        "pending": (0, 165, 255),       # orange: we need more frames
        "uncertain": (0, 215, 255),     # yellow: evidence is weak or mixed
        "compliant": (0, 200, 0),       # green: helmet accepted
        "violation": (0, 0, 255),       # red: no helmet accepted
    }

    # Motorcycles are background information, so draw them first, thin and
    # blue. Drawing them first means rider boxes stay on top and stay readable.
    for motorcycle_id, box in detections["motorcycle"].items():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (255, 150, 0), 1)
        cv2.putText(canvas, f"M{motorcycle_id}", (x1, max(18, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 150, 0), 1)

    for rider_id, rider_box in detections["rider"].items():
        # Skip riders we know nothing about: no motorcycle matched, or no head
        # seen yet. We deliberately draw nothing rather than a box saying
        # UNKNOWN. An empty space is honest; a label suggests we checked this
        # rider and reached a conclusion, which we did not.
        if rider_id not in rider_motorcycle or rider_id not in decisions:
            continue

        decision = decisions[rider_id]
        verdict = decision["verdict"]
        color = colors[verdict]
        x1, y1, x2, y2 = map(int, rider_box)

        # A violation gets a thicker box so it stands out on a busy frame.
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 3 if verdict == "violation" else 2)

        # The label carries the evidence, not just the answer:
        #   n = how many frames we have seen, a = how many of them agreed.
        label = f"R{rider_id} | {verdict.upper()}"
        if "agreement" in decision:
            label += f" | n={decision['observations']} a={decision['agreement']:.0%}"
        else:
            # A pending rider has no agreement figure yet.
            label += f" | n={decision['observations']}"
        cv2.putText(canvas, label, (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, color, 2)

        # Yellow line from the rider to the motorcycle we paired them with.
        if rider_id in rider_motorcycle:
            motorcycle_id = rider_motorcycle[rider_id]
            rider_point = tuple(map(int, center(rider_box)))
            motorcycle_point = tuple(map(int, center(detections["motorcycle"][motorcycle_id])))
            cv2.line(canvas, rider_point, motorcycle_point, (255, 255, 0), 2)

    return canvas


In [ ]:
# 🟩 RUN — Run the complete system over the whole clip
# A fresh model object for the full run.
pipeline_model = YOLO(str(MODEL_PATH))

# The three things we build up as the video plays:
# every observation we make about each rider...
observations_by_rider = defaultdict(list)
# ...the current answer for each rider...
decisions_by_rider = {}
# ...and one saved picture for each rider we report.
evidence = {}

# The clip is 500 frames, which is 20 seconds. We process all of it: most of
# the riders who end as pending or uncertain appear later in the clip, and
# they are an important part of the result. Set FAST_PREVIEW = True only if
# the session is running short of time.
FAST_PREVIEW = False
FRAME_LIMIT = 150 if FAST_PREVIEW else None

# Read the size and frame rate of the source clip, so the video we write
# matches it. A mismatch here makes the output play too fast or too slow.
source_info = cv2.VideoCapture(str(VIDEO_PATH))
SOURCE_FPS = source_info.get(cv2.CAP_PROP_FPS) or 25.0
SOURCE_WIDTH = int(source_info.get(cv2.CAP_PROP_FRAME_WIDTH))
SOURCE_HEIGHT = int(source_info.get(cv2.CAP_PROP_FRAME_HEIGHT))
source_info.release()

# Open the output video for writing.
RAW_ANNOTATED_VIDEO = OUTPUT_DIR / "helmet_detection_annotated_raw.mp4"
video_writer = cv2.VideoWriter(
    str(RAW_ANNOTATED_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"),
    SOURCE_FPS, (SOURCE_WIDTH, SOURCE_HEIGHT)
)
assert video_writer.isOpened(), "Could not open the annotated-video writer"

# Same tracking call as before, now over the whole video.
pipeline_stream = pipeline_model.track(
    source=str(VIDEO_PATH), tracker="bytetrack.yaml", persist=True, stream=True,
    classes=list(CLASS_NAMES), conf=0.25, imgsz=640, device=DEVICE, verbose=False
)

# Counts the frames we actually wrote, which is what we report at the end.
processed_frames = 0

# This loop runs for a few minutes on CPU, so print progress as we go.
# Otherwise the cell looks frozen and people restart it unnecessarily.
TOTAL_FRAMES = int(cv2.VideoCapture(str(VIDEO_PATH)).get(cv2.CAP_PROP_FRAME_COUNT))
expected = TOTAL_FRAMES if FRAME_LIMIT is None else min(FRAME_LIMIT, TOTAL_FRAMES)
print(f"Processing {expected} frames on {'GPU' if DEVICE != 'cpu' else 'CPU'}. "
      f"{'This takes about a minute.' if DEVICE != 'cpu' else 'On CPU this takes several minutes.'}")
started_at = time.time()

for frame_no, r in enumerate(pipeline_stream):
    # Stop early only if we set a frame limit above.
    if FRAME_LIMIT is not None and frame_no >= FRAME_LIMIT:
        break
    processed_frames = frame_no + 1

    # Every 50 frames, report speed and the time still to go.
    if processed_frames % 50 == 0:
        rate = processed_frames / (time.time() - started_at)
        remaining = (expected - processed_frames) / rate if rate else 0
        print(f"   {processed_frames}/{expected} frames  "
              f"({rate:.1f} fps, about {remaining:.0f}s left)")

    # --- Step 1: reshape this frame's output into our lookup tables.
    frame_detections, frame_confidences = unpack_tracked_result(r)

    # --- Step 2: work out who is on which motorcycle...
    rider_motorcycle = associate_riders_to_motorcycles(
        frame_detections["rider"], frame_detections["motorcycle"]
    )
    # ...and which head belongs to which rider.
    head_rider = associate_heads_to_riders(
        frame_detections["rider"], frame_detections["helmet"],
        frame_detections["no_helmet"], frame_confidences["helmet"],
        frame_confidences["no_helmet"]
    )

    # --- Step 3: record what we saw, and decide again for each rider.
    for rider_id, observation in head_rider.items():
        # Ignore a head unless its rider is also on a motorcycle. A pedestrian
        # on the footpath is not breaking a helmet law.
        if rider_id not in rider_motorcycle:
            continue

        # Add this frame's observation to that rider's history.
        observations_by_rider[rider_id].append(observation)

        # Decide again using the whole history, not just this frame. The
        # answer can change as more evidence arrives, which is the point.
        decision = decide_status(observations_by_rider[rider_id])
        decisions_by_rider[rider_id] = decision

        # The first time a rider becomes a violation, remember the frame.
        # The "not in evidence" check stops us saving the same rider again on
        # every following frame.
        if decision["verdict"] == "violation" and rider_id not in evidence:
            evidence[rider_id] = {
                "frame": frame_no,
                "decision": decision,
                "motorcycle_id": rider_motorcycle[rider_id],
            }

    # --- Step 4: draw this frame and add it to the output video.
    annotated = draw_pipeline_frame(
        r.orig_img, frame_detections, rider_motorcycle, decisions_by_rider
    )
    video_writer.write(annotated)

    # Keep a copy of the drawn frame for any violation confirmed just now.
    # We do this after drawing so the saved picture shows the red box.
    for rider_id, item in evidence.items():
        if item["frame"] == frame_no and "image" not in item:
            item["image"] = annotated.copy()

# Close the video file so it is complete and playable.
video_writer.release()

print(f"Processed {processed_frames} frames")
print(f"Riders we collected head evidence for: {len(observations_by_rider)}")
print(f"Violations reported: {len(evidence)}")
print("Raw annotated video:", RAW_ANNOTATED_VIDEO)


In [ ]:
# 🟩 RUN — Look at the evidence image for each violation
# This is the output a human reviewer would actually see. If the picture does
# not convince you, the system should not have reported it.
if evidence:
    # One row per violation found.
    fig, axes = plt.subplots(len(evidence), 1, figsize=(14, 7 * len(evidence)))
    # With a single violation, subplots returns one object instead of a list.
    # atleast_1d wraps it so the loop below works in both cases.
    axes = np.atleast_1d(axes)

    for ax, (rider_id, item) in zip(axes, evidence.items()):
        decision = item["decision"]
        ax.imshow(cv2.cvtColor(item["image"], cv2.COLOR_BGR2RGB))
        # The title carries the reasoning: which frame, how much the frames
        # agreed, and how confident the model was.
        ax.set_title(
            f"Rider {rider_id} — frame {item['frame']} — "
            f"agreement {decision['agreement']:.0%}, "
            f"confidence {decision['detector_confidence']:.2f}"
        )
        ax.axis("off")

        # Also save it as a file. In a real system this image is the evidence
        # attached to the case, so it must survive after the notebook closes.
        output_path = OUTPUT_DIR / f"violation_rider_{rider_id}_frame_{item['frame']}.jpg"
        cv2.imwrite(str(output_path), item["image"])
    plt.tight_layout()
    plt.show()
else:
    print("No rider met all three conditions, so nothing was reported.")


In [ ]:
# 🟩 RUN — Show every rider, including the ones we did not report
from collections import Counter

# First, how many riders ended in each of the four states?
print("How many riders ended in each state")
for verdict, count in Counter(d["verdict"] for d in decisions_by_rider.values()).most_common():
    print(f"   {verdict:<10} {count}")

# Now the detail, for the riders we saw most often.
print(f"\n{'rider':>6} {'frames':>7} {'mostly':>10} {'agree':>7} {'conf':>6}  answer")
print("-" * 58)
# Sort by number of observations, largest first: those carry the most evidence.
for rider_id, decision in sorted(decisions_by_rider.items(),
                                 key=lambda kv: -kv[1]["observations"])[:12]:
    # A pending rider has no majority or agreement yet, so print "-" instead.
    majority = decision.get("majority", "-")
    agreement = f"{decision['agreement']:.2f}" if "agreement" in decision else "-"
    confidence = f"{decision['detector_confidence']:.2f}" if "detector_confidence" in decision else "-"
    print(f"{rider_id:>6} {decision['observations']:>7} {majority:>10} "
          f"{agreement:>7} {confidence:>6}  {decision['verdict']}")

# The most interesting rows: the model mostly said "no helmet", and yet the
# system did not report a violation. Look at the reason on each line.
withheld = [(rid, d) for rid, d in decisions_by_rider.items()
            if d.get("majority") == "no_helmet" and d["verdict"] != "violation"]

print(f"\nRiders who looked like violations but were NOT reported: {len(withheld)}")
for rider_id, decision in sorted(withheld, key=lambda kv: -kv[1]["observations"]):
    print(f"   rider {rider_id}: seen in {decision['observations']} frames, "
          f"{decision['agreement']:.0%} agreed, average confidence "
          f"{decision['detector_confidence']:.2f}  ->  {decision['reason']}")


### 👀 OBSERVE — The answer the system refused to give

Look at the last part of the output, where riders with mostly *no_helmet*
frames were still **not** reported.

One rider there deserves your full attention. The model said *no helmet* in
about 96 out of every 100 frames, over more than a hundred frames. The system
still did not report a violation. The reason: the average confidence was about
0.67, and our rule asks for 0.70.

It missed by 0.03.

Looking at that rider, most people would say "yes, that person has no helmet".
The system does not say it, because the rule it was given was not met.

You can change the rule. Setting `MIN_CONFIDENCE = 0.65` will report this
rider immediately. Before you do, think about what else that change lets in:

- The same change also accepts every weaker case behind this one.
- A wrong violation has a real cost. Somebody gets blamed because of a blurred
  frame.
- A system that always gives an answer is not more correct. It is only more
  confident.

Try it, look at what else appears, and then decide what the right value is.
Choosing that number is engineering work. The model cannot choose it for you.


## Save and watch the video — Core

OpenCV writes the video in a format that most browsers cannot play. The next
cell converts it to H.264, which plays anywhere, and makes it smaller so it
can be shown inside the notebook.


In [ ]:
# 🟩 RUN — Convert and preview the annotated video
from IPython.display import Video, display

ANNOTATED_VIDEO_PATH = OUTPUT_DIR / "helmet_detection_annotated.mp4"

# The source clip is 2560x1440, so the annotated copy is large. The preview
# is scaled to 720p at crf 26: small enough to embed in the notebook and
# to play reliably in a browser, while the full-resolution file stays on
# disk. The full 500-frame clip lands near 12 MB, comfortably inside the
# 20 MB embed guard below.
conversion = subprocess.run(
    [
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", str(RAW_ANNOTATED_VIDEO),
        "-vf", "scale=1280:-2",
        "-c:v", "libx264", "-preset", "fast", "-crf", "26",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(ANNOTATED_VIDEO_PATH),
    ],
    capture_output=True, text=True,
)

if conversion.returncode != 0:
    print("H.264 conversion was unavailable; falling back to the OpenCV output.")
    print(conversion.stderr[-500:])
    ANNOTATED_VIDEO_PATH = RAW_ANNOTATED_VIDEO


def preview_video(path, limit_mb=20):
    """Embed the video only when it is small enough to be safe.

    An embedded video is base64-encoded into the notebook itself, so a
    50 MB file becomes roughly 70 MB of output. That can exhaust the
    browser tab and is refused by some notebook front ends.
    """
    size_mb = path.stat().st_size / 1e6
    print(f"Annotated video: {path}")
    print(f"Size: {size_mb:.2f} MB")
    if size_mb > limit_mb:
        print(f"Too large to embed ({size_mb:.0f} MB > {limit_mb} MB). "
              "Download it from the file browser to watch it locally.")
        return
    display(Video(str(path), embed=True))


preview_video(ANNOTATED_VIDEO_PATH)


### 👀 OBSERVE — How many riders actually got an answer?

Compare the two numbers printed above: how many riders were tracked, and how
many were reported as violations.

The difference is large, and that is correct behaviour. A rider seen for two
frames stays `pending`. A rider seen clearly but with mixed evidence stays
`uncertain`. Neither is a bug.

The next cell shows you exactly which riders were held back, and the reason
for each one.


### ⏭️ OPTIONAL — Download the files

This is switched off by default, so that **Run all** does not suddenly open a
download box on everyone's screen.


In [ ]:
# ⏭️ OPTIONAL — Download the results to your own computer
# Set this to True and run the cell again when you want the files.
DOWNLOAD_OUTPUTS = False

if DOWNLOAD_OUTPUTS:
    # google.colab is available only inside Colab, so import it here rather
    # than at the top of the notebook.
    from google.colab import files
    files.download(str(ANNOTATED_VIDEO_PATH))
else:
    print("Set DOWNLOAD_OUTPUTS = True when you want to download the video.")


# 7. Getting ready for the edge: export to ONNX — Core

*Time: 30 min.*

So far we have used the model in its PyTorch form. PyTorch is excellent for
training, but on a small device we usually do not want to carry Python and
PyTorch along with the model.

**ONNX** is a common file format for trained models. Many different tools can
load an ONNX file and run it. That makes your model easier to move to another
machine.

### What "edge deployment" means inside Colab

Colab is not an edge device. But it gives everyone in this room the same
environment, so we can all practise the same six steps:

1. export the trained model to ONNX,
2. load it with a runtime made for deployment,
3. force it to run on CPU, one frame at a time,
4. measure how long each frame takes,
5. check that the exported model still finds the same objects, and
6. save a small report with the model and the measurements.

These are the same steps you would follow on a Jetson, a Raspberry Pi or a
phone. Only the hardware changes. On a Jetson the next step after ONNX is
usually TensorRT, which your instructor can demonstrate.


In [ ]:
# 🟩 RUN — Convert the model to ONNX
# Load the weights again into a fresh object, so the export does not pick up
# any leftover state from our earlier tracking run.
export_model = YOLO(str(MODEL_PATH))

# export() writes a new file next to the original and returns its path.
#   format="onnx"    the file type we want
#   imgsz=640        fix the input size at 640x640
#   dynamic=False    fix the input shape instead of allowing any size. Fixed
#                    shapes let the runtime plan its work in advance, which is
#                    what small devices prefer.
#   simplify=True    tidy up the exported graph, removing redundant steps
#   opset=17         the ONNX version. Older devices may need a lower number.
onnx_path = Path(export_model.export(
    format="onnx",
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=17,
))

print("Exported model:", onnx_path)
# Compare the two files. The ONNX file is often larger, because the PyTorch
# file stores weights more compactly. File size alone tells you very little
# about speed, which is why we measure properly in the next cell.
print(f"PyTorch size: {MODEL_PATH.stat().st_size / 1e6:.2f} MB")
print(f"ONNX size:    {onnx_path.stat().st_size / 1e6:.2f} MB")


## How to measure speed honestly

We compare PyTorch and ONNX **on the same CPU**. Comparing PyTorch on a GPU
with ONNX on a CPU would only tell us that a GPU is faster than a CPU, which
we already know.

Two more rules we follow:

- The first few runs are thrown away. The first run of a model is always
  slower, because memory and other setup happen then. These are called
  **warm-up** runs.
- We report the **median** (the middle value) and the **p95** (the value that
  95 out of 100 runs stay below). A single fastest run is easy to get lucky
  with, and tells you nothing about the slow cases.

> These numbers describe the Colab machine you are using today. They do not
> describe a Jetson or a Raspberry Pi. Always measure again on the real device
> before you promise anyone a frame rate.


In [ ]:
# 🟩 RUN — Measure PyTorch and ONNX on the same CPU
# perf_counter is a high-resolution timer, better than time.time() for this.
import time


def synchronize_if_needed(device):
    """Wait for the GPU to finish before stopping the clock.

    A GPU runs work in the background: the Python line finishes before the
    GPU does. Without this wait we would measure how fast Python handed over
    the work, not how long the work took. On CPU nothing is needed.
    """
    if device != "cpu" and torch.cuda.is_available():
        torch.cuda.synchronize()


def benchmark_yolo(yolo_model, frame, device, warmup=3, runs=15):
    """Time one frame through the model, repeatedly, and summarise the result."""
    # Warm-up runs. The first calls are always slower because memory is being
    # allocated and code is being prepared. We throw these away.
    for _ in range(warmup):
        yolo_model.predict(frame, imgsz=640, conf=0.25, device=device, verbose=False)
    synchronize_if_needed(device)

    # Now the real measurements.
    latencies_ms = []
    for _ in range(runs):
        start = time.perf_counter()
        yolo_model.predict(frame, imgsz=640, conf=0.25, device=device, verbose=False)
        synchronize_if_needed(device)
        # Convert seconds to milliseconds, which is easier to read.
        latencies_ms.append((time.perf_counter() - start) * 1000)

    # The median is the middle value. Unlike the average, one very slow run
    # cannot drag it far off.
    median_ms = float(np.median(latencies_ms))
    return {
        "median_ms": median_ms,
        # p95: 95 runs out of 100 were faster than this. It shows the slow cases.
        "p95_ms": float(np.percentile(latencies_ms, 95)),
        # Frames per second, if every frame took the median time.
        "fps": 1000 / median_ms,
    }


# Build fresh objects for both sides, so neither carries state from earlier work.
pt_cpu_model = YOLO(str(MODEL_PATH))
onnx_model = YOLO(str(onnx_path), task="detect")

# Both on CPU. This is the comparison that is actually fair.
pt_cpu_results = benchmark_yolo(pt_cpu_model, sample_frame, "cpu")
onnx_cpu_results = benchmark_yolo(onnx_model, sample_frame, "cpu")

print(f"{'runtime':<18} {'median latency':>17} {'p95 latency':>14} {'FPS':>10}")
print("-" * 63)
print(f"{'PyTorch / CPU':<18} {pt_cpu_results['median_ms']:>14.1f} ms {pt_cpu_results['p95_ms']:>11.1f} ms {pt_cpu_results['fps']:>10.1f}")
print(f"{'ONNX / CPU':<18} {onnx_cpu_results['median_ms']:>14.1f} ms {onnx_cpu_results['p95_ms']:>11.1f} ms {onnx_cpu_results['fps']:>10.1f}")

# If this session has a GPU, print that number too, but keep it separate. It
# is interesting, but comparing it with the CPU rows above would only tell us
# that a GPU is faster than a CPU.
gpu_results = None
if torch.cuda.is_available():
    gpu_model = YOLO(str(MODEL_PATH))
    gpu_results = benchmark_yolo(gpu_model, sample_frame, 0)
    print(f"{'PyTorch / GPU':<18} {gpu_results['median_ms']:>14.1f} ms {gpu_results['p95_ms']:>11.1f} ms {gpu_results['fps']:>10.1f}")


### 👀 OBSERVE — What if ONNX is *slower*?

Look at your two rows. On many Colab machines the ONNX row is **slower** than
the PyTorch row. That is a real result, not a mistake by you.

Exporting a model makes it portable. It does not automatically make it faster.
Three reasons:

- PyTorch on CPU is already fast. It uses carefully hand-tuned code, so there
  is not much waste left for ONNX Runtime to remove.
- We run one frame at a time. Many speed tricks only pay off when you process
  a batch of images together.
- The exported model still uses the same 32-bit numbers as before. The big
  speed gains usually come from the *next* step: using smaller numbers
  (FP16 or INT8), or using a runtime built for one specific chip, such as
  TensorRT on a Jetson.

So what did we gain? A model file that no longer needs Python or PyTorch, and
that many other tools can load. Whether it is *fast enough* is a separate
question, and only measurement on the real device can answer it.

> Write down the number you actually measured, even when it is not the number
> you hoped for. A test that can only confirm what you already believe is not
> a test.


In [ ]:
# 🟩 RUN — Check that the exported model still sees the same objects
# Speed is worthless if the conversion quietly damaged the model. So run both
# versions on the same frame and compare the pictures side by side.
# Both run on CPU, so any difference comes from the conversion itself and not
# from different hardware doing the arithmetic differently.
pt_prediction = pt_cpu_model.predict(sample_frame, imgsz=640, conf=0.25, device="cpu", verbose=False)[0]
onnx_prediction = onnx_model.predict(sample_frame, imgsz=640, conf=0.25, device="cpu", verbose=False)[0]

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
axes[0].imshow(cv2.cvtColor(pt_prediction.plot(), cv2.COLOR_BGR2RGB))
axes[0].set_title(f"PyTorch: {len(pt_prediction.boxes)} objects")
axes[1].imshow(cv2.cvtColor(onnx_prediction.plot(), cv2.COLOR_BGR2RGB))
axes[1].set_title(f"ONNX: {len(onnx_prediction.boxes)} objects")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
# The counts should match, or be very close. A small difference can come from
# rounding. A large difference means something went wrong in the export.


## Run the exported model as a small edge application — Core

This is the part closest to a real deployment. The ONNX model reads the video
one frame at a time and runs on CPU. Later you would replace the video file
with a live camera, and the rest of the code would stay almost the same.

To keep the workshop moving, `EDGE_FRAME_LIMIT` is set to 100 frames. Set it
to `None` at home to run through the whole clip.


In [ ]:
# 🟩 RUN — Run the ONNX model like a small edge application
# How many frames to process. Set to None at home to run the whole clip.
EDGE_FRAME_LIMIT = 100

# We write the raw video first, then convert it for playback, exactly as before.
EDGE_RAW_VIDEO = OUTPUT_DIR / "edge_onnx_cpu_raw.mp4"
EDGE_VIDEO_PATH = OUTPUT_DIR / "edge_onnx_cpu.mp4"

# Open a video writer with the same size and frame rate as the source clip.
edge_writer = cv2.VideoWriter(
    str(EDGE_RAW_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"),
    SOURCE_FPS, (SOURCE_WIDTH, SOURCE_HEIGHT)
)
assert edge_writer.isOpened(), "Could not open the ONNX edge-video writer"

# We collect the time taken by every frame, to summarise at the end.
edge_latencies_ms = []

# stream=True gives us one frame at a time. On a small device with little
# memory this matters: the alternative keeps every processed frame in RAM.
edge_stream = onnx_model.predict(
    source=str(VIDEO_PATH), stream=True, imgsz=640, conf=0.25,
    device="cpu", verbose=False
)

for edge_frame_no, edge_result in enumerate(edge_stream):
    if EDGE_FRAME_LIMIT is not None and edge_frame_no >= EDGE_FRAME_LIMIT:
        break

    # Ultralytics times three separate stages for us:
    #   preprocess   resizing the frame and preparing it for the model
    #   inference    the model itself
    #   postprocess  turning raw output into boxes
    # A real application pays for all three, so we add them up. Quoting only
    # the inference time is how people end up promising a frame rate their
    # system cannot actually deliver.
    stage_times = edge_result.speed
    edge_latencies_ms.append(
        stage_times.get("preprocess", 0.0)
        + stage_times.get("inference", 0.0)
        + stage_times.get("postprocess", 0.0)
    )

    # Draw this frame's detections and write it into the output video.
    edge_writer.write(edge_result.plot())

# Close the file so it is complete on disk before we convert it.
edge_writer.release()

# Convert to a browser-friendly format, scaled down for a smaller file.
edge_conversion = subprocess.run(
    [
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(EDGE_RAW_VIDEO),
        "-vf", "scale=1280:-2",
        "-c:v", "libx264", "-preset", "fast", "-crf", "26",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(EDGE_VIDEO_PATH),
    ],
    capture_output=True, text=True,
)
# If the conversion failed, fall back to showing the unconverted file.
if edge_conversion.returncode != 0:
    EDGE_VIDEO_PATH = EDGE_RAW_VIDEO

edge_median_ms = float(np.median(edge_latencies_ms))
edge_p95_ms = float(np.percentile(edge_latencies_ms, 95))
print(f"Frames processed by ONNX Runtime: {len(edge_latencies_ms)}")
print(f"Median time per frame: {edge_median_ms:.1f} ms")
print(f"p95 time per frame: {edge_p95_ms:.1f} ms")
print(f"That is roughly {1000 / edge_median_ms:.1f} frames per second")
print("Output:", EDGE_VIDEO_PATH)
preview_video(EDGE_VIDEO_PATH)


## Write a deployment report — Core

A model file on its own is hard to trust six months later. Which video did we
test on? Which library versions? How fast was it, and on what machine?

The cell below saves all of that into one small JSON file. Keep this file next
to the model. It is what lets somebody else repeat your measurement.


In [ ]:
# 🟩 RUN — Save a report describing what we built and measured
import json
from datetime import datetime, timezone

# Everything a person would need to repeat this measurement later.
deployment_report = {
    # When, in UTC, so results from different people can be compared.
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "application": "helmet violation detection",
    "model": {
        "source": MODEL_PATH.name,
        "export": onnx_path.name,
        "input_size": 640,
        "source_size_mb": round(MODEL_PATH.stat().st_size / 1e6, 2),
        "onnx_size_mb": round(onnx_path.stat().st_size / 1e6, 2),
    },
    "runtime": {
        "name": "ONNX Runtime through Ultralytics",
        "device": "CPU",
        "batch_size": 1,
        # Version numbers matter: the same code on a different version can
        # give a different speed.
        "python": platform.python_version(),
        "ultralytics": ultralytics.__version__,
    },
    # The single-frame timings from the benchmark cell.
    "single_frame_benchmark": onnx_cpu_results,
    # The timings from the run over many frames.
    "stream_validation": {
        "frames": len(edge_latencies_ms),
        "median_pipeline_ms": round(edge_median_ms, 2),
        "p95_pipeline_ms": round(edge_p95_ms, 2),
        "approx_fps": round(1000 / edge_median_ms, 2),
    },
    # Write down what these numbers do NOT prove. A report that only lists
    # good news is not much use to the next person.
    "limitations": [
        "Colab CPU results do not represent Jetson performance",
        "Accuracy must be checked on a labelled test set after conversion",
        "Camera capture, power and heat are not measured in Colab",
    ],
}

REPORT_PATH = OUTPUT_DIR / "edge_deployment_report.json"
with open(REPORT_PATH, "w") as report_file:
    json.dump(deployment_report, report_file, indent=2)

print(json.dumps(deployment_report, indent=2))
print("Saved:", REPORT_PATH)


## Where quantization fits — Explore

**Quantization** means storing the model's numbers with less precision: 16-bit
or 8-bit instead of 32-bit. The model becomes smaller, and on the right
hardware it also becomes faster.

It is not a free speed-up, though:

- **On a Jetson:** TensorRT with FP16 is the usual first step, and it works
  well. INT8 needs extra work: you must supply sample images so the tool can
  calibrate.
- **On a normal CPU:** INT8 may be faster, slower, or not supported at all. It
  depends on the chip and the runtime. You must measure, and you must also
  check that accuracy did not drop.
- **In this workshop:** plain ONNX on CPU is the path that works for everyone.

This is a deliberate choice. Everyone here completes a real
export → run → measure → check cycle. The hardware-specific speed-up is shown
by the instructor, so that nobody's success depends on owning a Jetson.


### Exercise 4 — Explore

Fill in what you measured:

| Measurement | PyTorch | ONNX |
|---|---:|---:|
| Model size (MB) |  |  |
| Median time per frame (ms) |  |  |
| p95 time per frame (ms) |  |  |
| Frames per second |  |  |
| Objects found in the test frame |  |  |

Now answer:

1. Which one was faster on your machine today?
2. Did exporting change which objects were found?
3. A smaller model file is not always the better choice. Why not?


# If something goes wrong

| What you see | What to do |
|---|---|
| Download fails | Run the setup cell again. It retries by itself. If it still fails, ask the instructor for the copy on USB |
| Model or video missing | Run the setup cell again and read the paths it prints. A half-downloaded file is fetched again automatically |
| Colab disconnects | Reconnect, then run again from Setup. Files made during the session are lost |
| No GPU available | Carry on with CPU. Everything works, it is just slower |
| Only one violation found | That is the right answer for this clip. Run the cell that lists every rider to see who was held back, and why |
| Video is shorter than expected | `FAST_PREVIEW` is `True`. Set it to `False` and run the pipeline cell again |
| Video does not play | Check that FFmpeg said `OK` in the setup check. If it did, download the file and play it on your laptop |
| Export seems stuck | Wait. Do not run the cell again while it is still working |

If you get stuck and cannot catch up, switch to the instructor's finished
notebook and keep following the discussion. You can run your own copy later.


## Checklist for a real deployment

Before putting a system like this on a real road, you would need to tick all
of these:

- [ ] Decided the frame rate and the maximum delay you can accept
- [ ] Tested the camera position, resolution and different lighting
- [ ] Checked the confidence, matching and decision thresholds on real data
- [ ] Recorded how it fails: blur, hidden objects, crowded scenes, night time
- [ ] Kept `pending` and `uncertain` cases instead of forcing an answer
- [ ] Stored the evidence image together with the reason for the decision
- [ ] Written down model and library versions
- [ ] Checked accuracy again after export and after quantization
- [ ] Measured heat, power and memory after running for a long time
- [ ] Made sure a human checks every case before any action is taken


# Completion checklist

- [ ] I ran the model on a traffic frame and read its output.
- [ ] I saw track IDs stay with the same rider across frames.
- [ ] I matched riders to motorcycles, and heads to riders.
- [ ] I can explain `PENDING`, `UNCERTAIN`, `COMPLIANT` and `VIOLATION`.
- [ ] I produced a video with the results drawn on it.
- [ ] I exported the model to ONNX.
- [ ] I ran the ONNX model on CPU.
- [ ] I compared the speed of PyTorch and ONNX on the same CPU.
- [ ] I created `edge_deployment_report.json`.

## Five things to remember

1. A model finds objects. Turning objects into meaning is your job.
2. One frame is not evidence. Video decisions need several frames.
3. The best available match can still be a bad match, so check it.
4. A good system is allowed to say "not yet" or "I am not sure".
5. Speed claims mean nothing until you measure them on the real device.

### What to try after today

Run the whole video, ignore parts of the frame you do not care about, skip
blurred frames, crop and save the number plate, and — if you can get the
hardware — try TensorRT FP16 on a Jetson.
